# Unsupervised Hypersphere Fitting Algorithm

This notebook implements and adapts ideas from A classification method based on a cloud of spheres [ ]. The original work proposes constructing spherical regions to define decision boundaries between classes. In this project, the method is adapted for unsupervised anomaly detection using MVTec AD and DINOv2 embeddings.

The original paper differs from this setting in two important ways:

- Supervised boundary construction: 
The original algorithm uses both positive and negative classes when forming spheres. In this project, only normal training samples are available, so the sphere boundaries must be estimated without defective samples. To address this, candidate regions are initialised from local KNN density and then expanded using a constrained growth step.

- Connected manifold assumption:
The original method assumes that the data lie on a shared manifold and encourages connected spherical regions. This assumption is less appropriate here because DINOv2 produces 768-dimensional general-purpose embeddings. A single object category may occupy several disconnected regions in this embedding space. Therefore, the adapted algorithm does not require spheroids to be connected.

- Exclusive embedding assignment
The adapted algorithm enforces that each normal training embedding belongs to exactly one spheroid. During the growth stage, candidate regions are iteratively cleaned to ensure that previously assigned embeddings are not incorporated into newly generated spheroids. While geometric overlap between spheroids is permitted, overlap in embedding ownership is not. This results in a unique partitioning of the normal embedding space while still allowing neighbouring regions to share empty areas of the embedding space.

In [ ]:
import os

import torch
import sqlite3
import pandas as pd

import json

from datetime import datetime
from dataclasses import asdict

from pathlib import Path
import sys 

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import EMBEDS_DIR, EXPERIMENTS, RESULTS, DB_PATH, IMAGES, CACHES

EMBED_PATH = EMBEDS_DIR / "base_embeds"
EMBED_NAME = EMBED_PATH.stem

EXPERIMENTS_DIR = EXPERIMENTS / EMBED_NAME / "hypersphere"
RESULTS_DIR = RESULTS / EMBED_NAME

IMAGE_DIR = IMAGES / EMBED_NAME
CACHE_DIR = CACHES / EMBED_NAME

In [ ]:
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

In [7]:
cls_tokens = torch.load(EMBED_PATH/"cls.pt", weights_only=False)

In [8]:
conn = sqlite3.connect(DB_PATH)

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [11]:
%load_ext autoreload
%autoreload 2

from src.algorithims.hypersphere import CandidateCleaner, HypersphereEvaluator, HypersphereCover
from src.types import ExperimentConfig, AlgorithmResults

metadata = ExperimentConfig(
    K_frac=0.05,
    start_growth=1.05,
    min_growth=1
)

cleaner = CandidateCleaner()
cover = HypersphereCover(cleaner=cleaner)

eval = HypersphereEvaluator()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
time = datetime.now().strftime("%y-%m-%d_%H-%M-%S")

aurocs = {}

for category in categories:
    print("Running", category)
    outputs_dir = EXPERIMENTS_DIR / category / time
    os.makedirs(outputs_dir, exist_ok=True)

    train_mask = meta["split"] == "train"
    train_meta = meta[train_mask]
    test_meta = meta[~train_mask]

    train_cat_mask = train_meta["category"] == category

    train_emb = cls_tokens[train_mask]
    cat_emb = train_emb[train_cat_mask]

    spheres, spheres_df = cover.run(
        embeds=cat_emb, output_dir=outputs_dir,
        k_frac=metadata.K_frac,
        start_growth=metadata.start_growth, min_growth=metadata.min_growth
        )

    good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
    defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

    test_emb = cls_tokens[~train_mask]
    defect_test_emb = test_emb[defect_test_cat_mask]
    good_test_emb = test_emb[good_test_cat_mask]

    overlaps_df = eval.overlaps(cat_emb, spheres)
    overlaps_df.to_csv(outputs_dir / "overlaps.csv", index=False)

    good_any, good_counts = eval.inside_any_count(good_test_emb, spheres)
    defect_any, defect_counts = eval.inside_any_count(defect_test_emb, spheres)

    auroc = eval.scores(good_test_emb, defect_test_emb, spheres)
    aurocs[category] = auroc

    results = AlgorithmResults(
        config=metadata,
        category=category,
        n_shapes=len(spheres),
        auroc=auroc,
        normal_inside=int(good_any.sum()),
        defect_inside=int(defect_any.sum())
    )

    with open(outputs_dir / "metadata.json", "w") as f:
        json.dump(asdict(results), f)

aurocs_df = pd.DataFrame(aurocs.items(), columns=["Category", "AUROC"])


Running bottle
Running cable
Running capsule
Running carpet
Running grid
Running hazelnut
Running leather
Running metal_nut
Running pill
Running screw
Running tile
Running toothbrush
Running transistor
Running wood
Running zipper


In [15]:
df_roc_stats = pd.DataFrame({
    "mean": aurocs_df["AUROC"].mean(),
    "median": aurocs_df["AUROC"].median(),
    "std": aurocs_df["AUROC"].std(),
    "min_cat":  aurocs_df["Category"][aurocs_df["AUROC"].idxmin()],
    "min": aurocs_df["AUROC"].min(),
    "max_cat": aurocs_df["Category"][aurocs_df["AUROC"].idxmax()],
    "max": aurocs_df["AUROC"].max()
}, index=[0]).round(3)

df_roc_stats.to_csv(RESULTS_DIR / "hypersphere_auroc_stats.csv", index=False)

aurocs_df =aurocs_df.round(3)
aurocs_df.to_csv(RESULTS_DIR /"hypersphere_aurocs.csv", index=False)

df_roc_stats

,mean,median,std,min_cat,min,max_cat,max
0,0.89,0.919,0.083,screw,0.778,tile,0.997
